# Coupled Rooms FDN Example

Translation of `example_coupledRooms.m` to Python using FLAMO.

This example demonstrates coupled room acoustics using Feedback Delay Networks (FDN).

Based on ideas from:
> Das, O., Abel, J. S. & Canfield-Dafilou, E. K. *Delay Network Architectures For Room And Coupled Space Modeling*. in Proceedings of the 23rd International Conference on Digital Audio Effects (DAFx2020) (2020).

**Original MATLAB code:** (c) Sebastian Jiro Schlecht, Monday, 7. December 2020  
**Python translation:** Facundo Franchino, September 2025

In [ ]:
from collections import OrderedDict

import matplotlib.pyplot as plt
import numpy as np
import torch

# FLAMO imports
from flamo.processor import dsp, system

# pyFDN imports
from pyFDN.auxiliary.tiny_rotation_matrix import tiny_rotation_matrix

In [ ]:
# Set random seed for reproducibility (matches MATLAB rng(5))
torch.manual_seed(5)
np.random.seed(5)

## FDN Configuration

In [ ]:
# Parameters (matching MATLAB exactly)
fs = 48000
impulse_response_length = fs * 2  # 2 seconds
nfft = 16384

# FDN configuration
N = 12  # Total number of delay lines
N_per_room = N // 2  # 6 delay lines per room
num_input = 1
num_output = 2

# Device configuration
device = 'cuda' if torch.cuda.is_available() else 'cpu'
alias_decay_db = 0  # No anti-aliasing for exact reproduction

print(f"Using device: {device}")
print(f"FDN size: {N} delay lines ({N_per_room} per room)")

## Delay Lines

Exact delay values from MATLAB with `rng(5)`.

In [ ]:
# Exact delay values from MATLAB with rng(5)
delays_room1 = torch.tensor([411, 736, 403, 760, 544, 606], dtype=torch.float32)
delays_room2 = torch.tensor([2532, 2037, 1593, 1375, 1161, 2477], dtype=torch.float32)
delay_lengths = torch.cat([delays_room1, delays_room2])

print(f"Room 1 delays: {delays_room1.tolist()} samples")
print(f"Room 2 delays: {delays_room2.tolist()} samples")

## Feedback Matrix

Build the coupled feedback matrix using `tiny_rotation_matrix` for each room,
then combine them with a coupling parameter.

In [ ]:
# Coupling parameter (exact from MATLAB)
coupling = 0.3

# Generate feedback matrices using tinyRotationMatrix
# Matches MATLAB: A1 = tinyRotationMatrix(N/2,12); A2 = tinyRotationMatrix(N/2,12);
A1 = tiny_rotation_matrix(N_per_room, 12).float()
A2 = tiny_rotation_matrix(N_per_room, 12).float()

# Compute matrix square roots using eigenvalue decomposition
def matrix_sqrt(A):
    """sqrtm(A) = V * sqrt(D) * V^(-1) where A = V * D * V^(-1)"""
    eigenvals, eigenvecs = torch.linalg.eig(A.to(torch.complex64))
    sqrt_eigenvals = torch.sqrt(eigenvals)
    return torch.real(eigenvecs @ torch.diag(sqrt_eigenvals) @ torch.linalg.inv(eigenvecs)).float()

A1_sqrt = matrix_sqrt(A1)
A2_sqrt = matrix_sqrt(A2)

# Build the exact coupled feedback matrix from MATLAB
cos_c = torch.cos(torch.tensor(coupling))
sin_c = torch.sin(torch.tensor(coupling))

# Create the block matrix structure
feedback_matrix = torch.zeros(N, N)
feedback_matrix[:N_per_room, :N_per_room] = cos_c * A1
feedback_matrix[:N_per_room, N_per_room:] = sin_c * torch.matmul(A1_sqrt, A2_sqrt)
feedback_matrix[N_per_room:, :N_per_room] = -sin_c * torch.matmul(A2_sqrt, A1_sqrt)
feedback_matrix[N_per_room:, N_per_room:] = cos_c * A2

# Verify orthogonality
ortho_check = torch.matmul(feedback_matrix.T, feedback_matrix)
max_deviation = torch.max(torch.abs(ortho_check - torch.eye(N)))
print(f"Feedback matrix orthogonality check (max deviation from I): {max_deviation:.2e}")

## Build FDN using FLAMO

In [ ]:
# Input gain: source only in first room (matches MATLAB exactly)
input_gain = dsp.Gain(
    size=(N, num_input),
    nfft=nfft,
    requires_grad=False,
    alias_decay_db=alias_decay_db,
    device=device,
)
input_gain_values = torch.zeros(N, num_input)
input_gain_values[:N_per_room, :] = 1.0  # First 6 delays get input
input_gain.assign_value(input_gain_values)

# Output gain: block diagonal [ones(1,6), zeros; zeros, ones(1,6)]
output_gain = dsp.Gain(
    size=(num_output, N),
    nfft=nfft,
    requires_grad=False,
    alias_decay_db=alias_decay_db,
    device=device,
)
output_gain_values = torch.zeros(num_output, N)
output_gain_values[0, :N_per_room] = 1.0  # Room 1 to left channel
output_gain_values[1, N_per_room:] = 1.0   # Room 2 to right channel
output_gain.assign_value(output_gain_values)

In [ ]:
# Create delay lines
delays = dsp.parallelDelay(
    size=(N,),
    max_len=int(delay_lengths.max()),
    nfft=nfft,
    isint=True,
    requires_grad=False,
    alias_decay_db=alias_decay_db,
    device=device,
)
delays.assign_value(delays.sample2s(delay_lengths.int()))

# Create mixing matrix with the exact feedback matrix
mixing_matrix = dsp.Matrix(
    size=(N, N),
    nfft=nfft,
    matrix_type="random",  # We'll override with our values
    requires_grad=False,
    alias_decay_db=alias_decay_db,
    device=device,
)
mixing_matrix.assign_value(feedback_matrix)

In [ ]:
# T60 values from MATLAB (at 1kHz)
shortT60 = torch.tensor([0.5, 0.5, 0.55, 0.575, 0.525, 0.375, 0.275, 0.2, 0.175, 0.175])
longT60 = torch.tensor([4.0, 4.0, 4.4, 4.6, 4.2, 3.0, 2.2, 1.6, 1.4, 1.4])

# For simplicity, use frequency-independent attenuation at 1kHz band
attenuation = dsp.parallelGain(
    size=(N,),
    nfft=nfft,
    requires_grad=False,
    alias_decay_db=alias_decay_db,
    device=device,
)

def t60_to_gain_per_sample(t60, fs):
    """Convert T60 to gain coefficient per sample"""
    return 10 ** (-3 / (t60 * fs))

# Use the 1kHz band T60 values
short_t60_1khz = shortT60[4].item()  # 0.525 seconds
long_t60_1khz = longT60[4].item()    # 4.2 seconds

attenuation_values = torch.zeros(N)
# Room 1 (short T60)
g_short = t60_to_gain_per_sample(short_t60_1khz, fs)
for i in range(N_per_room):
    attenuation_values[i] = g_short ** delay_lengths[i]

# Room 2 (long T60)
g_long = t60_to_gain_per_sample(long_t60_1khz, fs)
for i in range(N_per_room, N):
    attenuation_values[i] = g_long ** delay_lengths[i]

attenuation.assign_value(attenuation_values)

print(f"Room 1 T60 (1kHz): {short_t60_1khz} s")
print(f"Room 2 T60 (1kHz): {long_t60_1khz} s")

In [ ]:
# Create feedback path
feedback = system.Series(
    OrderedDict({
        "mixing_matrix": mixing_matrix,
        "attenuation": attenuation
    })
)

# Create recursion (feedback loop)
feedback_loop = system.Recursion(fF=delays, fB=feedback)

# Complete FDN
fdn = system.Series(
    OrderedDict({
        "input_gain": input_gain,
        "feedback_loop": feedback_loop,
        "output_gain": output_gain,
    })
)

# Create shell with FFT/iFFT
input_layer = dsp.FFT(nfft)
output_layer = dsp.iFFT(nfft)

model = system.Shell(
    core=fdn,
    input_layer=input_layer,
    output_layer=output_layer
)

print("FDN model built successfully!")

## Generate Impulse Response

In [ ]:
# Generate impulse response
with torch.no_grad():
    impulse = torch.zeros(1, nfft, 1)
    impulse[0, 0, 0] = 1.0
    ir = model(impulse).squeeze().cpu().numpy()

    # Correct shape
    if ir.ndim == 1:
        ir = ir.reshape(-1, 1)

    # Trim to desired length
    ir = ir[:impulse_response_length, :]

print(f"IR shape: {ir.shape}")
print(f"IR duration: {ir.shape[0] / fs:.2f} seconds")

## Visualization

In [ ]:
fig = plt.figure(figsize=(15, 5))

# Plot 1: Impulse responses
ax1 = plt.subplot(1, 3, 1)
samples = np.arange(len(ir))
ax1.plot(samples, ir[:, 0], label='Short Room', alpha=0.7, linewidth=0.5)
if ir.shape[1] > 1:
    ax1.plot(samples, ir[:, 1] - 2, label='Long Room', alpha=0.7, linewidth=0.5)
ax1.set_xlabel('Samples')
ax1.set_ylabel('Amplitude')
ax1.set_title('Coupled Rooms Impulse Response')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim([0, len(ir)])

# Plot 2: Feedback matrix
ax2 = plt.subplot(1, 3, 2)
im = ax2.imshow(feedback_matrix.numpy(), cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
ax2.set_title('Feedback Matrix')
ax2.set_xlabel('Column')
ax2.set_ylabel('Row')
ax2.axhline(y=5.5, color='black', linewidth=1, linestyle='--', alpha=0.5)
ax2.axvline(x=5.5, color='black', linewidth=1, linestyle='--', alpha=0.5)

# Plot 3: Energy decay curves
ax3 = plt.subplot(1, 3, 3)
t = np.arange(len(ir)) / fs
edc1 = np.cumsum(ir[::-1, 0]**2)[::-1]
edc1_db = 10 * np.log10(edc1 / (edc1[0] + 1e-12))
ax3.plot(t, edc1_db, label='Short Room', alpha=0.8)

if ir.shape[1] > 1:
    edc2 = np.cumsum(ir[::-1, 1]**2)[::-1]
    edc2_db = 10 * np.log10(edc2 / (edc2[0] + 1e-12))
    ax3.plot(t, edc2_db, label='Long Room', alpha=0.8)

ax3.set_xlabel('Time (s)')
ax3.set_ylabel('Energy (dB)')
ax3.set_title('Energy Decay Curves')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_xlim([0, min(2, len(ir)/fs)])
ax3.set_ylim([-60, 0])

plt.tight_layout()
plt.show()